# PPO Tutorial-Style Training

Este notebook e o cockpit de treino e experimentacao do `PPO`. Segue agora a mesma logica do `DQN`: treino estruturado no codigo e notebook focado em iterar variantes, correr treino e analisar a run ativa. Para apresentar a melhor versao, usa o `05_ppo_best_model_showcase.ipynb`.


## Passo 1: Setup


In [ ]:
from __future__ import annotations

import copy
import json
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'connect4_rl').exists():
        ROOT = candidate
        break
else:
    raise RuntimeError('Nao encontrei a raiz do repositorio com a pasta connect4_rl.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from connect4_rl.agents.baselines import MinimaxAgent, RandomAgent, StrongHeuristicAgent, WeakHeuristicAgent
from connect4_rl.config import load_config
from connect4_rl.experiments import build_agent_from_run, evaluate_against_agent, find_best_run
from connect4_rl.experiments.ppo_notebook_variants import VARIANT_SPECS, apply_variant_to_config
from connect4_rl.experiments.ppo_training import build_tutorial_ppo_lessons, evaluate_match_summary, train_ppo_self_play


## Passo 2: Configuracao do treino

O perfil `quick` usa o budget de notebook para iteracao rapida. O perfil `full` aproxima-se mais do regime final. O campo `experiment_variant` permite testar hipoteses controladas sem mexer no codigo base.


In [ ]:
NOTEBOOK_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
training_profile = 'quick'  # 'quick' ou 'full'
experiment_variant = 'baseline'  # 'baseline', 'robust_selection', 'safer_self_play', 'wider_policy', 'anti_strong', 'wider_safer_strong', 'final_push', 'final_push_midlevel_bc', 'final_push_minimax2_tune'
seed = 42
run_training = True

CONFIG = load_config(str(ROOT / 'config.yaml'))
training_config = apply_variant_to_config(copy.deepcopy(CONFIG), experiment_variant)
training_config.global_.seed = seed

if training_profile == 'quick':
    training_config.ppo.episodes = CONFIG.notebook_settings.quick_test.episodes
    training_config.ppo.eval_interval = CONFIG.notebook_settings.quick_test.eval_interval
    training_config.ppo.eval_games = CONFIG.notebook_settings.quick_test.eval_games
else:
    training_config.ppo.episodes = max(training_config.ppo.episodes, 720)
    training_config.ppo.eval_interval = 30
    training_config.ppo.eval_games = 24

variant_spec = VARIANT_SPECS[experiment_variant]
run_name = f'ppo_tutorial_{training_profile}_{experiment_variant}_seed_{seed}'
OUTPUTS = ROOT / 'notebooks' / 'ppo' / 'outputs'
OUTPUTS.mkdir(parents=True, exist_ok=True)
checkpoint_dir = OUTPUTS / run_name
lessons = build_tutorial_ppo_lessons(training_config.ppo.episodes, profile=training_config.ppo.curriculum_profile)
training_plan = {
    'run_name': run_name,
    'training_profile': training_profile,
    'experiment_variant': experiment_variant,
    'variant_description': variant_spec['description'],
    'episodes': training_config.ppo.episodes,
    'eval_interval': training_config.ppo.eval_interval,
    'eval_games': training_config.ppo.eval_games,
    'lessons': [lesson.name for lesson in lessons],
    'checkpoint_dir': str(checkpoint_dir),
}
training_plan


## Passo 3: Treino do pipeline PPO


In [ ]:
trained_agent = None
trained_metrics = None

if run_training:
    trained_agent, trained_metrics = train_ppo_self_play(training_config, checkpoint_dir=checkpoint_dir)
    {
        'curriculum_name': trained_metrics.curriculum_name,
        'experiment_variant': experiment_variant,
        'bootstrap_summary': trained_metrics.bootstrap_summary,
        'lessons': [item['lesson_name'] for item in trained_metrics.lesson_summaries],
        'final_checkpoint_path': trained_metrics.best_checkpoint_path,
        'focus_checkpoint_path': trained_metrics.best_vs_strong_checkpoint_path,
        'last_eval': trained_metrics.evaluation[-1] if trained_metrics.evaluation else {},
        'num_policy_updates': len(trained_metrics.policy_losses),
    }
else:
    print('Treino nao executado nesta sessao. O notebook vai tentar carregar a melhor run guardada.')


## Passo 4: Escolher a run ativa


In [ ]:
active_metrics = trained_metrics
active_agent = trained_agent
active_run_name = run_name if run_training else None
focus_checkpoint_path = None

if active_metrics is not None:
    active_run_name = run_name
else:
    best_run = find_best_run(OUTPUTS, 'ppo')
    if best_run is None:
        raise RuntimeError('Nao encontrei runs PPO em outputs/. Corre primeiro o treino neste notebook.')
    active_run_name = best_run.metrics_path.parent.name
    active_metrics = type('MetricsProxy', (), best_run.data)()
    active_agent = build_agent_from_run(best_run, root=ROOT, device=NOTEBOOK_DEVICE)

focus_checkpoint_path = getattr(active_metrics, 'best_vs_strong_checkpoint_path', '') or getattr(active_metrics, 'best_checkpoint_path', '')
lesson_summaries = getattr(active_metrics, 'lesson_summaries', [])
evaluation = getattr(active_metrics, 'evaluation', [])
episode_rewards = getattr(active_metrics, 'episode_rewards', [])

print({
    'run_name': active_run_name,
    'bootstrap_summary': getattr(active_metrics, 'bootstrap_summary', {}),
    'lessons': [item['lesson_name'] for item in lesson_summaries],
    'num_evaluations': len(evaluation),
    'focus_checkpoint_path': focus_checkpoint_path,
    'best_vs_strong_win_rate': round(float(getattr(active_metrics, 'best_vs_strong_win_rate', 0.0)), 4),
    'best_vs_strong_draw_rate': round(float(getattr(active_metrics, 'best_vs_strong_draw_rate', 0.0)), 4),
    'reward_first_20_mean': round(sum(episode_rewards[:20]) / max(len(episode_rewards[:20]), 1), 4),
    'reward_last_20_mean': round(sum(episode_rewards[-20:]) / max(len(episode_rewards[-20:]), 1), 4),
})
lesson_summaries


## Passo 5: Curvas de avaliacao


In [ ]:
evaluation = list(getattr(active_metrics, 'evaluation', []))
phase_summary = getattr(active_metrics, 'phase_summary', [])
policy_losses = [float(value) for value in getattr(active_metrics, 'policy_losses', [])]
value_losses = [float(value) for value in getattr(active_metrics, 'value_losses', [])]
entropies = [float(value) for value in getattr(active_metrics, 'entropies', [])]
rewards = [float(value) for value in getattr(active_metrics, 'episode_rewards', [])]

def moving_average(values, window):
    if len(values) < window:
        return values
    return [statistics.fmean(values[max(0, idx - window + 1): idx + 1]) for idx in range(len(values))]

if evaluation:
    eval_episodes = [int(item['episode']) for item in evaluation]
    eval_outcome = [float(item.get('eval_mean_outcome', 0.0)) for item in evaluation]
    vs_random = [float(item.get('vs_random_win_rate', 0.0)) for item in evaluation]
    vs_weak = [float(item.get('vs_weak_heuristic_win_rate', 0.0)) for item in evaluation]
    vs_strong = [float(item.get('vs_strong_heuristic_win_rate', 0.0)) for item in evaluation]
    vs_minimax_1 = [float(item.get('vs_minimax_1_win_rate', 0.0)) for item in evaluation]
    vs_minimax_2 = [float(item.get('vs_minimax_2_win_rate', 0.0)) for item in evaluation]
    vs_previous = [float(item.get('vs_previous_win_rate', 0.0)) for item in evaluation]
    best_vs_strong_idx = max(range(len(vs_strong)), key=lambda idx: vs_strong[idx])
    print('Best vs strong checkpoint')
    print({
        'episode': eval_episodes[best_vs_strong_idx],
        'lesson': evaluation[best_vs_strong_idx].get('lesson_name'),
        'vs_strong': round(vs_strong[best_vs_strong_idx], 4),
        'vs_strong_draw': round(float(evaluation[best_vs_strong_idx].get('vs_strong_draw_rate', 0.0)), 4),
        'eval_mean_outcome': round(eval_outcome[best_vs_strong_idx], 4),
        'vs_random': round(vs_random[best_vs_strong_idx], 4),
        'vs_weak': round(vs_weak[best_vs_strong_idx], 4),
        'vs_minimax_1': round(vs_minimax_1[best_vs_strong_idx], 4),
        'vs_minimax_2': round(vs_minimax_2[best_vs_strong_idx], 4),
        'vs_previous': round(vs_previous[best_vs_strong_idx], 4),
    })

    print('Evaluation checkpoints')
    for item in evaluation:
        print({
            'episode': int(item['episode']),
            'lesson': item.get('lesson_name'),
            'eval_mean_outcome': round(float(item.get('eval_mean_outcome', 0.0)), 4),
            'vs_random': round(float(item.get('vs_random_win_rate', 0.0)), 4),
            'vs_weak': round(float(item.get('vs_weak_heuristic_win_rate', 0.0)), 4),
            'vs_minimax_1': round(float(item.get('vs_minimax_1_win_rate', 0.0)), 4),
            'vs_minimax_2': round(float(item.get('vs_minimax_2_win_rate', 0.0)), 4),
            'vs_strong': round(float(item.get('vs_strong_heuristic_win_rate', 0.0)), 4),
            'vs_strong_draw': round(float(item.get('vs_strong_draw_rate', 0.0)), 4),
            'vs_previous': round(float(item.get('vs_previous_win_rate', 0.0)), 4),
        })

    print('Lesson summaries')
    for item in lesson_summaries:
        print({
            'lesson': item.get('lesson_name'),
            'best_score': round(float(item.get('best_score', 0.0)), 4),
            'best_vs_strong': round(float(item.get('best_vs_strong_win_rate', 0.0)), 4),
            'best_vs_strong_draw': round(float(item.get('best_vs_strong_draw_rate', 0.0)), 4),
        })

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].plot(eval_episodes, vs_random, marker='o', label='vs random')
    axes[0, 0].plot(eval_episodes, vs_weak, marker='o', label='vs weak')
    axes[0, 0].plot(eval_episodes, vs_minimax_1, marker='o', label='vs minimax_1')
    axes[0, 0].plot(eval_episodes, vs_minimax_2, marker='o', label='vs minimax_2')
    axes[0, 0].plot(eval_episodes, vs_strong, marker='o', label='vs strong')
    axes[0, 0].scatter([eval_episodes[best_vs_strong_idx]], [vs_strong[best_vs_strong_idx]], color='red', s=80, label='best vs strong')
    axes[0, 0].set_title('Win rate por checkpoint')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Win rate')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(eval_episodes, eval_outcome, marker='o', color='purple', label='eval mean outcome')
    axes[0, 1].plot(eval_episodes, vs_previous, marker='o', color='gray', label='vs previous')
    axes[0, 1].set_title('Outcome e estabilidade')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(rewards, alpha=0.35, label='Reward por episodio')
    axes[1, 0].plot(moving_average(rewards, 20), linewidth=2, label='Media movel (20)')
    axes[1, 0].set_title('Recompensa de treino')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(policy_losses, label='policy loss')
    axes[1, 1].plot(value_losses, label='value loss')
    if entropies:
        axes[1, 1].plot(entropies, label='entropy')
    axes[1, 1].set_title('Sinais de otimizacao')
    axes[1, 1].set_xlabel('Update step')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Ainda nao ha checkpoints de avaliacao para mostrar.')


## Passo 6: Avaliacao final contra referencias


In [ ]:
final_eval = {
    'vs_random': evaluate_match_summary(active_agent, lambda game_idx: RandomAgent(seed=10_000 + game_idx), games=80),
    'vs_weak': evaluate_match_summary(active_agent, lambda game_idx: WeakHeuristicAgent(seed=20_000 + game_idx), games=80),
    'vs_minimax_1': evaluate_match_summary(active_agent, lambda game_idx: MinimaxAgent(depth=1, seed=25_000 + game_idx), games=80),
    'vs_minimax_2': evaluate_match_summary(active_agent, lambda game_idx: MinimaxAgent(depth=2, seed=27_000 + game_idx), games=80),
    'vs_strong': evaluate_match_summary(active_agent, lambda game_idx: StrongHeuristicAgent(seed=30_000 + game_idx), games=80),
}
final_eval
